**Cell 1**

# 02 — Combine Aggregated Classes into the Unified Split Dataset

Input (from notebook 01):
```
data/aggregated/<class>/
├── images/  (200 jpgs)
├── labels/  (200 YOLO txts, class_id = 0)
└── info.json
```

This notebook merges all four classes into one unified dataset with a **stratified 70/20/10 split** and remaps every `class_id` from `0` to the global index (`projector=0`, `whiteboard=1`, `fire_extinguisher=2`, `door_sign=3`):

```
data/dataset/
├── images/{train,val,test}/
├── labels/{train,val,test}/
└── data.yaml
```

In [ ]:
# Cell 2 — install dependencies for this notebook
%pip install -q pandas pillow matplotlib pyyaml

In [ ]:
# Cell 3 — setup
import random, shutil
from pathlib import Path
import pandas as pd
import yaml
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

random.seed(42)
CLASSES = ['projector', 'whiteboard', 'fire_extinguisher', 'door_sign']
GLOBAL_ID = {c: i for i, c in enumerate(CLASSES)}
SPLIT_RATIOS = {'train': 0.70, 'val': 0.20, 'test': 0.10}

AGG = Path('../data/aggregated')
DST = Path('../data/dataset')
for s in ['train', 'val', 'test']:
    (DST / 'images' / s).mkdir(parents=True, exist_ok=True)
    (DST / 'labels' / s).mkdir(parents=True, exist_ok=True)

**Cell 4**

## 2.1 Discover aggregated pairs per class

In [ ]:
# Cell 5 — list image+label pairs from data/aggregated/<class>/
pairs_per_class = {}
for c in CLASSES:
    imgs = sorted((AGG / c / 'images').glob('*.jpg'))
    pairs = [(img, AGG / c / 'labels' / (img.stem + '.txt')) for img in imgs]
    pairs = [(i, l) for i, l in pairs if l.exists()]
    pairs_per_class[c] = pairs

pd.DataFrame([(c, len(p)) for c, p in pairs_per_class.items()],
             columns=['class', 'available_pairs'])

**Cell 6**

## 2.2 Stratified 70 / 20 / 10 split, per class
Splitting per class guarantees each class is represented in every split.

In [ ]:
# Cell 7 — assign each pair to a split
assignments = []
for c, pairs in pairs_per_class.items():
    pool = pairs[:]; random.shuffle(pool)
    n = len(pool)
    n_tr = int(n * SPLIT_RATIOS['train'])
    n_va = int(n * SPLIT_RATIOS['val'])
    for img, lbl in pool[:n_tr]:             assignments.append((c, img, lbl, 'train'))
    for img, lbl in pool[n_tr:n_tr+n_va]:    assignments.append((c, img, lbl, 'val'))
    for img, lbl in pool[n_tr+n_va:]:        assignments.append((c, img, lbl, 'test'))

plan = pd.DataFrame(assignments, columns=['class', 'img', 'lbl', 'split'])
plan.groupby(['split', 'class']).size().unstack(fill_value=0).reindex(['train','val','test'])

**Cell 8**

## 2.3 Copy into split folders; rewrite class_id to the global index

In [ ]:
# Cell 9 — copy + remap
counter = {(c, s): 0 for c in CLASSES for s in ['train', 'val', 'test']}
manifest = []
for row in plan.itertuples():
    counter[(row._1, row.split)] += 1
    idx = counter[(row._1, row.split)]
    stem = f'{row._1}_{row.split}_{idx:04d}'

    dst_img = DST / 'images' / row.split / f'{stem}.jpg'
    dst_lbl = DST / 'labels' / row.split / f'{stem}.txt'
    shutil.copy2(row.img, dst_img)

    gid = GLOBAL_ID[row._1]
    remapped = []
    for line in row.lbl.read_text().strip().splitlines():
        parts = line.split()
        if len(parts) != 5:
            continue
        _, xc, yc, w, h = parts
        remapped.append(f'{gid} {xc} {yc} {w} {h}')
    dst_lbl.write_text('\n'.join(remapped) + ('\n' if remapped else ''))

    manifest.append({'filename': dst_img.name, 'class': row._1,
                     'global_id': gid, 'split': row.split, 'boxes': len(remapped)})

mf = pd.DataFrame(manifest)
mf.to_csv(DST / 'manifest.csv', index=False)
mf.groupby(['split', 'class']).agg(images=('filename','count'), boxes=('boxes','sum'))

**Cell 10**

## 2.4 Structural validation
Every image has a matching label; every row has 5 numeric fields; class_id in `[0,3]`; coords in `[0,1]`.

In [ ]:
# Cell 11 — validate labels across all splits
issues = []
for split in ['train', 'val', 'test']:
    img_dir = DST / 'images' / split; lbl_dir = DST / 'labels' / split
    for img in img_dir.glob('*.jpg'):
        lbl = lbl_dir / (img.stem + '.txt')
        if not lbl.exists():
            issues.append((split, img.name, 'missing_label')); continue
        for ln, line in enumerate(lbl.read_text().strip().splitlines(), 1):
            parts = line.split()
            if len(parts) != 5:
                issues.append((split, img.name, f'bad_row_{ln}')); continue
            cid, *coords = parts
            if int(cid) not in range(len(CLASSES)):
                issues.append((split, img.name, f'bad_class_{cid}'))
            if any(not (0 <= float(x) <= 1) for x in coords):
                issues.append((split, img.name, f'coords_out_of_range_row_{ln}'))

pd.DataFrame(issues, columns=['split', 'file', 'issue']).head(20)

**Cell 12**

## 2.5 Visual spot-check — confirm the class remap is correct

In [ ]:
# Cell 13 — overlay 3 train images per class
def draw(ax, img_path, lbl_path):
    im = Image.open(img_path); W, H = im.size
    ax.imshow(im); ax.set_title(img_path.name, fontsize=8); ax.axis('off')
    for line in lbl_path.read_text().strip().splitlines():
        cid, xc, yc, w, h = map(float, line.split())
        x = (xc - w/2) * W; y = (yc - h/2) * H
        ax.add_patch(patches.Rectangle((x, y), w*W, h*H, fill=False, linewidth=1.5, edgecolor='lime'))
        ax.text(x, y-3, CLASSES[int(cid)], color='lime', fontsize=7)

train_img = DST / 'images' / 'train'; train_lbl = DST / 'labels' / 'train'
fig, axes = plt.subplots(len(CLASSES), 3, figsize=(12, 4 * len(CLASSES)))
for row, c in enumerate(CLASSES):
    picks = random.sample(list(train_img.glob(f'{c}_*.jpg')), 3)
    for ax, p in zip(axes[row], picks):
        draw(ax, p, train_lbl / (p.stem + '.txt'))
plt.tight_layout(); plt.show()

**Cell 14**

## 2.6 Emit `data.yaml` for Ultralytics

In [ ]:
# Cell 15 — write data.yaml
cfg = {
    'path': str(DST.resolve()),
    'train': 'images/train',
    'val':   'images/val',
    'test':  'images/test',
    'names': {i: c for i, c in enumerate(CLASSES)},
}
(DST / 'data.yaml').write_text(yaml.safe_dump(cfg, sort_keys=False))
print((DST / 'data.yaml').read_text())

**Cell 16**

### Acceptance criteria
- Cell 9 shows ≈140/40/20 images per class across train/val/test
- Cell 11 returns an empty issues table
- Cell 13 shows correct labels overlaid for all four classes
- `data/dataset/data.yaml` exists

Proceed to **notebook 03** for dataset-health diagnostics.